# Vie-GameEmo — Inference & Demo

**Notebook này thực hiện:**
1. Load checkpoint đã train
2. Chạy inference trên clip mới (upload hoặc YouTube)
3. Hiển thị kết quả: nhãn cảm xúc, confidence scores, attention weights
4. LLM explanation (tùy chọn, chậm hơn)
5. Ablation: so sánh 4 LLM setups

**Yêu cầu:**
- Accelerator: GPU T4 (cần cho LLM explanation)
- Input dataset: checkpoint từ notebook Training
- Internet: BẬT (để tải model nếu cần)

**Cách thêm checkpoint:**  
Upload `vie_gameemo_checkpoints.zip` lên Kaggle Dataset → thêm vào notebook inputs

In [ ]:
# ============================================================
# CELL 1 — Môi trường
# ============================================================
import os, sys, gc, json, shutil
from pathlib import Path
import torch

IS_KAGGLE = os.path.exists('/kaggle')
WORKING = '/kaggle/working' if IS_KAGGLE else '/tmp/vie-gameemo'
os.makedirs(WORKING, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | {gpu.total_memory / 1e9:.1f} GB VRAM')
print(f'Device: {device}')

In [ ]:
# ============================================================
# CELL 2 — CẤU HÌNH
# ============================================================

# --- Checkpoint ---
# Option A: từ Kaggle input dataset
CKPT_INPUT = '/kaggle/input/vie-gameemo-checkpoints/checkpoints/perception_best.pt'
# Option B: đường dẫn cục bộ trong working
CKPT_LOCAL = os.path.join(WORKING, 'checkpoints/perception_best.pt')

CHECKPOINT = CKPT_INPUT if os.path.exists(CKPT_INPUT) else CKPT_LOCAL
print(f'Checkpoint: {CHECKPOINT}')
if not os.path.exists(CHECKPOINT):
    print('⚠️  Không tìm thấy checkpoint — hãy chạy notebook Training trước')

# --- Input clip ---
# 'youtube'  : tải từ URL
# 'upload'   : file đã upload vào Kaggle
# 'dataset'  : từ Kaggle input dataset
CLIP_SOURCE = 'youtube'
CLIP_URL    = ''   # URL YouTube (khi CLIP_SOURCE = 'youtube')
CLIP_PATH   = ''   # đường dẫn file (khi CLIP_SOURCE = 'upload' hoặc 'dataset')

# --- LLM Explanation ---
# 'none'  : chỉ classification, không dùng LLM (nhanh)
# 'llm1'  : Post-hoc explainer (Qwen2.5-7B, không train)
# 'llm2'  : Co-Reasoner (text inputs)
# 'llm3'  : VLM end-to-end (Qwen2.5-VL-7B)
# 'llm4'  : RLVR-trained (cần adapter từ train_rlvr)
LLM_SETUP = 'llm1'

# Model cho LLM
# 'Qwen/Qwen2.5-7B-Instruct'   → T4 OK
# 'Qwen/Qwen2.5-1.5B-Instruct' → nhanh hơn, ít VRAM
# 'Qwen/Qwen3-8B'              → Qwen3, hỗ trợ thinking mode
LLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
LLM_QUANT = '4bit'

# --- Fusion config (phải khớp với model đã train) ---
FUSION_TYPE = 'conv_attention_4m'

print(f'\nLLM_SETUP : {LLM_SETUP}')
print(f'LLM_MODEL : {LLM_MODEL}')
print(f'FUSION    : {FUSION_TYPE}')

In [ ]:
# ============================================================
# CELL 3 — Cài thư viện
# ============================================================
!pip install -q \
    transformers>=4.45.0 \
    accelerate>=0.34.0 \
    peft>=0.13.0 \
    bitsandbytes>=0.43.0 \
    faster-whisper>=1.0.3 \
    librosa>=0.10.1 \
    soundfile \
    mediapipe>=0.10.14 \
    yt-dlp \
    pydantic>=2.8.0 \
    pyyaml \
    scikit-learn \
    matplotlib

if LLM_SETUP == 'llm3':
    !pip install -q qwen-vl-utils

print('Done')

In [ ]:
# ============================================================
# CELL 4 — Setup project
# ============================================================
import subprocess

PROJECT_DIR = os.path.join(WORKING, 'vie-gameemo-skeleton')
PROJECT_INPUT = '/kaggle/input/vie-gameemo-code'

if os.path.exists(PROJECT_INPUT):
    if not os.path.exists(PROJECT_DIR):
        shutil.copytree(PROJECT_INPUT, PROJECT_DIR)
elif not os.path.exists(PROJECT_DIR):
    GITHUB_URL = 'https://github.com/YOUR_USERNAME/vie-gameemo-skeleton.git'
    subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, PROJECT_DIR])

SRC_DIR = os.path.join(PROJECT_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Đường dẫn
DATA_DIR = os.path.join(WORKING, 'data')
PROC_DIR = os.path.join(DATA_DIR, 'processed')
FEAT_DIR = os.path.join(DATA_DIR, 'features')
CKPT_DIR = os.path.join(WORKING, 'checkpoints')
for d in [DATA_DIR, PROC_DIR, FEAT_DIR, CKPT_DIR,
           os.path.join(PROC_DIR, 'audios'),
           os.path.join(PROC_DIR, 'frames')]:
    os.makedirs(d, exist_ok=True)

print(f'Project: {PROJECT_DIR}')
print(f'SRC   : {SRC_DIR}')

## Bước 1 — Load Model

In [ ]:
# ============================================================
# CELL 5 — Load Fusion + Classifier
# ============================================================
from vie_gameemo.classifiers.mlp import EmotionClassifier
from vie_gameemo.fusion import get_fusion
from vie_gameemo.training.perception import load_checkpoint

LABEL_NAMES = ['hype', 'tilted', 'focused', 'disappointed', 'shocked', 'amused', 'neutral']
LABEL_EMOJI = {
    'hype': '🔥', 'tilted': '😤', 'focused': '🎯',
    'disappointed': '😞', 'shocked': '😱', 'amused': '😄', 'neutral': '😐'
}

if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(f'Checkpoint không tồn tại: {CHECKPOINT}\nChạy notebook Training trước.')

fusion_model = get_fusion(
    FUSION_TYPE,
    d_model=768, n_modalities=4, n_conv_blocks=4,
    kernel_size=3, align_to='audio', return_attention=True,
).to(device)
classifier = EmotionClassifier(768, 256, 7, dropout=0.0).to(device)

load_checkpoint(Path(CHECKPOINT), fusion_model, classifier)
fusion_model.eval()
classifier.eval()
print(f'✅ Model loaded từ: {CHECKPOINT}')

total_params = sum(p.numel() for p in fusion_model.parameters()) + \
               sum(p.numel() for p in classifier.parameters())
print(f'   Total params: {total_params/1e6:.2f}M')

## Bước 2 — Chuẩn bị clip

In [ ]:
# ============================================================
# CELL 6 — Lấy clip
# ============================================================
clip_path = None

if CLIP_SOURCE == 'youtube':
    if not CLIP_URL:
        print('⚠️  CLIP_URL rỗng — điền URL YouTube vào CELL 2')
    else:
        out_path = os.path.join(DATA_DIR, 'raw_videos/inference_clip.mp4')
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        print(f'Downloading: {CLIP_URL[:80]}...')
        r = subprocess.run([
            'yt-dlp', CLIP_URL,
            '-f', 'best[height<=720]',
            '--merge-output-format', 'mp4',
            '-o', out_path,
            '--no-playlist',
        ])
        if r.returncode == 0:
            clip_path = out_path
            print(f'✅ Downloaded: {out_path}')
        else:
            print('❌ Tải thất bại')

elif CLIP_SOURCE in ('upload', 'dataset'):
    if CLIP_PATH and os.path.exists(CLIP_PATH):
        clip_path = CLIP_PATH
        print(f'Clip: {clip_path}')
    else:
        print(f'⚠️  Không tìm thấy: {CLIP_PATH}')

# Demo: nếu vẫn không có clip, tạo video giả để test pipeline
if clip_path is None or not os.path.exists(clip_path):
    print('\nTạo dummy video để test pipeline...')
    dummy_path = os.path.join(DATA_DIR, 'raw_videos/dummy.mp4')
    os.makedirs(os.path.dirname(dummy_path), exist_ok=True)
    r = subprocess.run([
        'ffmpeg', '-y', '-f', 'lavfi',
        '-i', 'color=c=blue:s=640x360:d=5',
        '-f', 'lavfi', '-i', 'sine=frequency=440:duration=5',
        '-c:v', 'libx264', '-c:a', 'aac',
        dummy_path
    ], capture_output=True)
    if r.returncode == 0:
        clip_path = dummy_path
        print(f'✅ Dummy video: {dummy_path} (5 giây, màu xanh + tone 440Hz)')
        print('⚠️  Kết quả sẽ không có nghĩa với dummy video — hãy dùng clip thực')
    else:
        print('❌ ffmpeg không tạo được dummy video')

if clip_path:
    stat = os.stat(clip_path)
    print(f'Clip size: {stat.st_size / 1e6:.1f} MB')

In [ ]:
# ============================================================
# CELL 7 — Tiền xử lý clip (audio + frames)
# ============================================================
from vie_gameemo.preprocess.demux import extract_audio, extract_frames

clip_id = 'inference_clip'
audio_path = Path(os.path.join(PROC_DIR, 'audios', f'{clip_id}.wav'))
frames_dir = Path(os.path.join(PROC_DIR, 'frames', clip_id))
frames_dir.mkdir(parents=True, exist_ok=True)

print('Tách audio...')
extract_audio(Path(clip_path), audio_path)
print(f'  Audio: {audio_path} ({audio_path.stat().st_size/1e3:.0f} KB)')

print('Trích xuất frames...')
n_frames = extract_frames(Path(clip_path), frames_dir, target_fps=2)
print(f'  Frames: {n_frames} frames → {frames_dir}')

frame_paths = sorted(frames_dir.glob('*.jpg'))

# Xem preview frames
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

if frame_paths:
    n_preview = min(4, len(frame_paths))
    fig, axes = plt.subplots(1, n_preview, figsize=(4*n_preview, 3))
    if n_preview == 1:
        axes = [axes]
    indices = [int(i * (len(frame_paths)-1) / (n_preview-1)) for i in range(n_preview)] if n_preview > 1 else [0]
    for ax, idx in zip(axes, indices):
        img = mpimg.imread(str(frame_paths[idx]))
        ax.imshow(img)
        ax.set_title(f'Frame {idx}')
        ax.axis('off')
    plt.suptitle('Preview frames')
    plt.tight_layout()
    plt.show()

## Bước 3 — Trích xuất Features & Inference

In [ ]:
# ============================================================
# CELL 8 — Trích xuất features (inline, không cache)
# ============================================================
features = {}

# Audio
print('Encoding audio (AST)...')
from vie_gameemo.encoders.audio_ast import ASTAudioEncoder
audio_enc = ASTAudioEncoder()
features['audio'] = audio_enc.encode(audio_path)  # (1, 64, 768)
del audio_enc; gc.collect(); torch.cuda.empty_cache()
print(f'  audio: {tuple(features["audio"].shape)}')

# Context
print('Encoding context (ViT-ImageNet)...')
from vie_gameemo.encoders.context_vit import ContextEncoder
ctx_enc = ContextEncoder()
features['context'] = ctx_enc.encode(frame_paths)  # (1, T, 768)
del ctx_enc; gc.collect(); torch.cuda.empty_cache()
print(f'  context: {tuple(features["context"].shape)}')

# Face
print('Encoding face (ViT-FER)...')
from vie_gameemo.encoders.face_vit import FaceEncoder
face_enc = FaceEncoder()
features['face'], features['has_face'] = face_enc.encode(frame_paths)
del face_enc; gc.collect(); torch.cuda.empty_cache()
print(f'  face: {tuple(features["face"].shape)} | has_face={features["has_face"]}')

# Text (transcript)
print('Encoding text (XLM-R)...')
from vie_gameemo.encoders.text_xlmr import XLMRTextEncoder
text_enc = XLMRTextEncoder()
# Nếu không có transcript, dùng chuỗi rỗng
features['text'] = text_enc.encode('')  # (1, T, 768)
del text_enc; gc.collect(); torch.cuda.empty_cache()
print(f'  text: {tuple(features["text"].shape)}')

print('\n✅ Feature extraction xong')

In [ ]:
# ============================================================
# CELL 9 — Inference: Fusion + Classifier
# ============================================================
import time

t0 = time.perf_counter()

audio_t   = features['audio'].unsqueeze(0).to(device)    # (1, 1, 64, 768)
face_t    = features['face'].unsqueeze(0).to(device)
context_t = features['context'].unsqueeze(0).to(device)
text_t    = features['text'].unsqueeze(0).to(device)
has_face_t = torch.tensor([[features['has_face']]], dtype=torch.bool, device=device)

# Thêm batch dim nếu thiếu (features đã có batch dim từ encoders)
# audio shape từ encoder: (1, 64, 768) → cần (B=1, T=64, D=768) OK

with torch.no_grad():
    result = fusion_model(audio_t, face_t, context_t, text_t, has_face=has_face_t)
    if isinstance(result, tuple):
        fused, attn_weights = result
    else:
        fused, attn_weights = result, None

    logits = classifier(fused)        # (1, 7)
    probs  = torch.softmax(logits, dim=-1)[0]   # (7,)

elapsed_ms = (time.perf_counter() - t0) * 1000
pred_idx   = int(probs.argmax().item())
pred_label = LABEL_NAMES[pred_idx]
pred_conf  = float(probs[pred_idx].item())

print(f'=== KẾT QUẢ ===')
print(f'Cảm xúc dự đoán: {LABEL_EMOJI.get(pred_label,"")} {pred_label.upper()}')
print(f'Confidence      : {pred_conf:.1%}')
print(f'Latency         : {elapsed_ms:.1f} ms')
print()
print('Tất cả nhãn:')
sorted_probs = sorted(zip(LABEL_NAMES, probs.cpu().tolist()), key=lambda x: -x[1])
for label, prob in sorted_probs:
    bar = '█' * int(prob * 30)
    marker = '◀' if label == pred_label else ''
    print(f'  {LABEL_EMOJI.get(label,"")} {label:<15}: {bar:<30} {prob:.3f} {marker}')

In [ ]:
# ============================================================
# CELL 10 — Visualize kết quả
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bar chart confidence ---
ax1 = axes[0]
labels_sorted = [x[0] for x in sorted_probs]
probs_sorted  = [x[1] for x in sorted_probs]
emojis = [LABEL_EMOJI.get(l, '') for l in labels_sorted]
colors = ['#FF6B35' if l == pred_label else '#BDC3C7' for l in labels_sorted]

bars = ax1.barh(
    [f'{e} {l}' for e, l in zip(emojis, labels_sorted)],
    probs_sorted, color=colors, edgecolor='white'
)
ax1.set_xlabel('Probability')
ax1.set_title(f'Emotion Prediction: {LABEL_EMOJI.get(pred_label,"")} {pred_label.upper()} ({pred_conf:.1%})',
              fontsize=13, fontweight='bold')
ax1.set_xlim(0, 1)
for bar, prob in zip(bars, probs_sorted):
    ax1.text(min(prob + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
             f'{prob:.3f}', va='center', fontsize=9)

# --- Attention weights (nếu có) ---
ax2 = axes[1]
if attn_weights is not None:
    attn = attn_weights[0].cpu().numpy()  # (4,)
    modality_names = ['Audio\n(AST)', 'Face\n(ViT-FER)', 'Context\n(ViT)', 'Text\n(XLM-R)']
    bars2 = ax2.bar(modality_names, attn, color=['#3498DB', '#E74C3C', '#2ECC71', '#9B59B6'])
    ax2.set_title('Modality Attention Weights', fontsize=12)
    ax2.set_ylabel('Attention Weight')
    ax2.set_ylim(0, 1)
    for bar, val in zip(bars2, attn):
        ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val:.3f}', ha='center', fontsize=10)
else:
    ax2.text(0.5, 0.5, 'Attention weights\nnot available\n(set return_attention=True)',
             ha='center', va='center', transform=ax2.transAxes, fontsize=11, color='gray')
    ax2.set_title('Modality Attention Weights', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(WORKING, 'inference_result.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Kết quả đã lưu → inference_result.png')

## Bước 4 — LLM Explanation (tùy chọn)

In [ ]:
# ============================================================
# CELL 11 — LLM Reasoning
# ============================================================
if LLM_SETUP == 'none':
    print('LLM_SETUP=none — bỏ qua explanation')
    print('Để bật: đặt LLM_SETUP = "llm1" ở CELL 2')
else:
    print(f'Loading LLM ({LLM_SETUP}): {LLM_MODEL} [{LLM_QUANT}]...')

    # Build evidence dict
    evidence = {
        'label': pred_label,
        'face_aus': 'N/A (không có OpenFace)',
        'visual_objective': 'N/A',
        'audio_tone': 'N/A',
        'transcript': '',
        'game_context': 'Game stream',
        'pitch_hz': 200,
        'rms_db': -20,
        'shout': False,
    }

    llm_out = None
    try:
        if LLM_SETUP == 'llm1':
            from vie_gameemo.llm.llm1_explainer import LLM1Explainer
            from vie_gameemo.utils.config import load_config
            template = (
                'Streamer được dự đoán đang ở trạng thái: {label}.\n\n'
                'Bằng chứng:\n- Khuôn mặt: {face_aus}\n- Bối cảnh: {game_context}\n'
                '- Giọng: pitch={pitch_hz}Hz, rms={rms_db}dB, shout={shout}\n'
                '- Transcript: "{transcript}"\n\n'
                'Giải thích ngắn gọn (dưới 100 từ) vì sao streamer ở trạng thái này:'
            )
            llm = LLM1Explainer(
                model_name=LLM_MODEL,
                prompt_template=template,
                quantization=LLM_QUANT,
            )

        elif LLM_SETUP == 'llm2':
            from vie_gameemo.llm.llm2_coreasoner import LLM2CoReasoner
            llm = LLM2CoReasoner(model_name=LLM_MODEL, quantization=LLM_QUANT)

        elif LLM_SETUP == 'llm3':
            from vie_gameemo.llm.llm3_vlm import LLM3VLMEndToEnd
            llm = LLM3VLMEndToEnd(quantization=LLM_QUANT)
            evidence['frame_paths'] = frame_paths[:4]
            evidence['audio_path'] = audio_path

        elif LLM_SETUP == 'llm4':
            from vie_gameemo.llm.llm4_rlvr import LLM4RLVR
            rlvr_adapter = os.path.join(CKPT_DIR, 'llm4_rlvr')
            adapter = rlvr_adapter if os.path.exists(rlvr_adapter) else None
            llm = LLM4RLVR(base_model=LLM_MODEL, adapter_path=adapter, quantization=LLM_QUANT)

        llm_out = llm.reason(evidence)
        llm.unload()
        del llm
        gc.collect(); torch.cuda.empty_cache()

    except Exception as e:
        print(f'⚠️  LLM error: {e}')

    if llm_out:
        print(f'\n=== LLM EXPLANATION ({LLM_SETUP.upper()}) ===')
        print(f'Format valid: {llm_out.format_valid}')
        print(f'Answer      : {LABEL_EMOJI.get(llm_out.answer, "")} {llm_out.answer}')
        print(f'\nReasoning:')
        print(llm_out.reasoning)

## Bước 5 — So sánh 4 LLM Setups (Ablation)

In [ ]:
# ============================================================
# CELL 12 — So sánh LLM setups trên cùng một clip
# (chậm: mỗi setup cần load/unload riêng)
# ============================================================
RUN_LLM_ABLATION = False   # Đặt True để so sánh tất cả 4 setups

if RUN_LLM_ABLATION:
    results = {}

    # LLM-1
    print('--- LLM-1: Post-hoc Explainer ---')
    try:
        from vie_gameemo.llm.llm1_explainer import LLM1Explainer
        template = 'Streamer đang ở trạng thái {label}. Giải thích ngắn dựa trên bằng chứng.'
        llm1 = LLM1Explainer(model_name=LLM_MODEL, prompt_template=template, quantization=LLM_QUANT)
        out1 = llm1.reason({**evidence, 'label': pred_label})
        results['llm1'] = out1
        llm1.unload(); del llm1
        gc.collect(); torch.cuda.empty_cache()
        print(f'  Answer: {out1.answer} | Format: {out1.format_valid}')
    except Exception as e:
        results['llm1'] = None
        print(f'  ⚠️  {e}')

    # LLM-2
    print('--- LLM-2: Co-Reasoner ---')
    try:
        from vie_gameemo.llm.llm2_coreasoner import LLM2CoReasoner
        llm2 = LLM2CoReasoner(model_name=LLM_MODEL, quantization=LLM_QUANT)
        out2 = llm2.reason(evidence)
        results['llm2'] = out2
        llm2.unload(); del llm2
        gc.collect(); torch.cuda.empty_cache()
        print(f'  Answer: {out2.answer} | Format: {out2.format_valid}')
    except Exception as e:
        results['llm2'] = None
        print(f'  ⚠️  {e}')

    # LLM-3
    print('--- LLM-3: VLM End-to-End ---')
    try:
        from vie_gameemo.llm.llm3_vlm import LLM3VLMEndToEnd
        llm3 = LLM3VLMEndToEnd(quantization=LLM_QUANT)
        out3 = llm3.reason({**evidence, 'frame_paths': frame_paths[:4], 'audio_path': audio_path})
        results['llm3'] = out3
        llm3.unload(); del llm3
        gc.collect(); torch.cuda.empty_cache()
        print(f'  Answer: {out3.answer} | Format: {out3.format_valid}')
    except Exception as e:
        results['llm3'] = None
        print(f'  ⚠️  {e}')

    # LLM-4
    print('--- LLM-4: RLVR ---')
    try:
        from vie_gameemo.llm.llm4_rlvr import LLM4RLVR
        rlvr_adapter = os.path.join(CKPT_DIR, 'llm4_rlvr')
        llm4 = LLM4RLVR(base_model=LLM_MODEL,
                          adapter_path=rlvr_adapter if os.path.exists(rlvr_adapter) else None,
                          quantization=LLM_QUANT)
        out4 = llm4.reason(evidence)
        results['llm4'] = out4
        llm4.unload(); del llm4
        gc.collect(); torch.cuda.empty_cache()
        print(f'  Answer: {out4.answer} | Format: {out4.format_valid}')
    except Exception as e:
        results['llm4'] = None
        print(f'  ⚠️  {e}')

    # Bảng so sánh
    print('\n=== SO SÁNH 4 LLM SETUPS ===')
    print(f'{"Setup":<8} {"Answer":<20} {"Format":<8} {"Reasoning (50 chars)"}')
    print('-' * 80)
    for key, out in results.items():
        if out:
            rshort = out.reasoning[:50].replace('\n', ' ') if out.reasoning else 'N/A'
            print(f'{key:<8} {out.answer:<20} {str(out.format_valid):<8} {rshort}')
        else:
            print(f'{key:<8} {"ERROR":<20} {"N/A":<8}')
else:
    print('RUN_LLM_ABLATION=False')
    print('Đặt RUN_LLM_ABLATION = True để so sánh tất cả 4 LLM setups (cần ~30 phút)')

In [ ]:
# ============================================================
# CELL 13 — Lưu kết quả
# ============================================================
output = {
    'clip_path': str(clip_path),
    'predicted_label': pred_label,
    'confidence': pred_conf,
    'class_scores': {l: float(p) for l, p in zip(LABEL_NAMES, probs.cpu().tolist())},
    'attn_weights': attn_weights[0].cpu().tolist() if attn_weights is not None else None,
    'llm_setup': LLM_SETUP,
    'llm_reasoning': llm_out.reasoning if 'llm_out' in dir() and llm_out else None,
    'llm_answer': llm_out.answer if 'llm_out' in dir() and llm_out else None,
}

result_path = os.path.join(WORKING, 'inference_result.json')
with open(result_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'✅ Kết quả lưu → {result_path}')
print(json.dumps(output, ensure_ascii=False, indent=2))

---
## Eval chính thức trên toàn bộ test split

In [ ]:
# ============================================================
# CELL 14 — Eval trên test split (nếu có annotations)
# ============================================================
ANNOT_INPUT_EVAL = '/kaggle/input/vie-gameemo-annotations/annotations'
ANNOT_DIR_EVAL   = ANNOT_INPUT_EVAL if os.path.exists(ANNOT_INPUT_EVAL) else None

if ANNOT_DIR_EVAL and os.path.exists(ANNOT_DIR_EVAL):
    from torch.utils.data import DataLoader
    from vie_gameemo.data.dataset import VieGameEmoDataset, collate_fn, make_splits
    from vie_gameemo.training.perception import evaluate
    from vie_gameemo.evaluation.metrics import compute_metrics
    from vie_gameemo.evaluation.per_genre import per_genre_metrics, format_genre_table

    LABEL2IDX = {l: i for i, l in enumerate(LABEL_NAMES)}
    SPLITS_PATH = Path(ANNOT_DIR_EVAL) / 'splits.json'
    if not SPLITS_PATH.exists():
        make_splits(Path(ANNOT_DIR_EVAL), output_path=SPLITS_PATH, seed=42)

    FEAT_DIR_EVAL = '/kaggle/input/vie-gameemo-checkpoints/features'
    if not os.path.exists(FEAT_DIR_EVAL):
        FEAT_DIR_EVAL = FEAT_DIR

    test_ds = VieGameEmoDataset(
        Path(ANNOT_DIR_EVAL), Path(FEAT_DIR_EVAL),
        split='test_id', splits_path=SPLITS_PATH, label2idx=LABEL2IDX,
    )
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False,
                             num_workers=2, collate_fn=collate_fn)

    metrics = evaluate(fusion_model, classifier, test_loader, device, n_classes=7)
    print('=== TEST_ID EVALUATION ===')
    print(f'  Accuracy   : {metrics["accuracy"]:.4f}')
    print(f'  Macro F1   : {metrics["macro_f1"]:.4f}')
    print(f'  Weighted F1: {metrics["weighted_f1"]:.4f}')
    print(f'  UAR        : {metrics["uar"]:.4f}')
else:
    print('Không có annotation dataset → bỏ qua eval chính thức')
    print('Thêm dataset annotations vào notebook inputs để eval')